In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls "/content/drive/MyDrive"

 17491237056674837552505831388117.jpg  'DEEP_s_Resume AI&ML (1)-1 (3).pdf'
 AdmitCard-260410853562_copy.pdf       'DEEP_s_Resume AI&ML (1)-1.pdf'
'Adobe Scan Apr 16, 2026.pdf'	       'DEEP_s_Resume AI&ML.pdf'
 Certificate.pdf		       'Fee Deep 4th sem Samarth eGov.pdf'
'Colab Notebooks'		        Garbage_Detection.ipynb
 Deep_Resume-1.pdf		        Garbage_My_Dataset.yolov8.zip
 Deep_Resume.pdf		       'Maths for DSA and CP.gdoc'
'DEEP_s_Resume AI&ML (1)-1 (1).pdf'     Synopsis.pdf
'DEEP_s_Resume AI&ML (1)-1 (2).pdf'


In [ ]:

!find "/content/drive/MyDrive" -iname "*.zip"

/content/drive/MyDrive/Colab Notebooks/Pothole_D_yolov8.zip
/content/drive/MyDrive/Colab Notebooks/garbage_Detection.zip
/content/drive/MyDrive/Garbage_My_Dataset.yolov8.zip


In [ ]:
ZIP_PATH = "/content/drive/MyDrive/Garbage_My_Dataset.yolov8.zip"

In [ ]:
!unzip -q "$ZIP_PATH" -d /content/My_garbage_Detection

replace /content/My_garbage_Detection/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: All


In [ ]:
import os

for root, dirs, files in os.walk("/content/My_garbage_Detection"):
    print(root)

/content/My_garbage_Detection
/content/My_garbage_Detection/test
/content/My_garbage_Detection/test/labels
/content/My_garbage_Detection/test/images
/content/My_garbage_Detection/train
/content/My_garbage_Detection/train/labels
/content/My_garbage_Detection/train/images


In [ ]:
!find /content/My_garbage_Detection -name "*.yaml"

/content/My_garbage_Detection/data.yaml


In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.6 MB/s eta 0:00:00


In [ ]:
from pathlib import Path

train_images = list(
    Path("//content/My_garbage_Detection/train/images").glob("*")
)

print("Training Images:", len(train_images))

Training Images: 135


In [ ]:
import os

In [ ]:
%%writefile /content/data.yaml
train: /content/My_garbage_Detection/train/images
val: /content/My_garbage_Detection/test/images
test: /content/My_garbage_Detection/test/images

nc: 5
names: ['biodegradable', 'metal', 'paper', 'plastic','leaf']

Overwriting /content/data.yaml


In [ ]:
from ultralytics import YOLO


model = YOLO("yolov8s.pt")

model.train(
    data="/content/data.yaml",
    epochs=50,
    imgsz=135,
    batch=16
)

import os

os.makedirs("runs", exist_ok=True)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=135, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspect

In [ ]:
metrics = model.val()

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1924.2±0.0 MB/s, size: 132.4 KB)
val: Scanning /content/My_garbage_Detection/test/labels.cache... 1 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1/1 466.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 15.3it/s 0.1s
                   all          1          4     0.0247        0.5     0.0582    0.00942
         biodegradable          1          4     0.0247        0.5     0.0582    0.00942
Speed: 0.5ms preprocess, 54.2ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to /content/runs/detect/val
mAP50: 0.058235294117647066
mAP50-95: 0.009417366946778711
Precision: 0.024689161243360656
Recall: 0.5


In [ ]:
!cp \
/content/runs/detect/train-3/weights/best.pt \
/content/drive/MyDrive/garbage_Detection.pt

### Object Detection on Video

To perform object detection on a video, you can pass the path to your video file as the `source` argument to the `predict` method. The results, including detected objects, will be saved to a video file in the `runs/detect/predict` directory (or a similarly named directory if multiple predictions are made).

In [ ]:
from ultralytics import YOLO

# Load the trained model
best_model = YOLO(
    "/content/runs/detect/train-3/weights/best.pt" # Corrected path to the best.pt file
)

# Replace 'path/to/your/video.mp4' with the actual path to your video file
# For example, if your video is in Google Drive, you'd use something like
# '/content/drive/MyDrive/your_video.mp4'
video_path = "/content/Garbage_V1.mp4"

# Perform prediction on the video
# The results (video with bounding boxes) will be saved in a new directory
# like 'runs/detect/predictX' (where X is an incrementing number)
results = best_model.predict(
    source=video_path,
    conf=0.25, # Confidence threshold for detections
    save=True, # Save the output video with detections
    show=False, # Set to True if you want to display the video during prediction (might be slow or require specific environment)
    show_labels=False # This will remove class names from bounding boxes
)

print(f"Video processing complete. Results saved in: {results[0].save_dir}")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/557) /content/Garbage_V1.mp4: 160x96 2 biodegradables, 65.0ms
video 1/1 (frame 2/557) /content/Garbage_V1.mp4: 160x96 4 biodegradables, 14.7ms
video 1/1 (frame 3/557) /content/Garbage_V1.mp4: 160x96 3 biodegradables, 11.7ms
video 1/1 (frame 4/557) /content/Garbage_V1.mp4: 160x96 2 biodegradables, 12.2ms
video 1/1 (frame 5/557) /content/Garbage_V1.mp4: 160x96 1 biodegradable, 10.7ms
video 1/1 (frame 6/557) /content/Garbage_V1.mp4: 160x9